# 03 — Paired-bootstrap test (synthetic, known truth)

Three groups (A, B, C) with **known true** intergroup correlations $\rho_{AB}$, $\rho_{AC}$, $\rho_{BC}$. Validates the paired-bootstrap test by checking:

1. Under $H_0$ ($\rho_{AB} = \rho_{AC} = \rho_{BC}$), the recentered-null p-value is uniformly distributed and rejects at $\sim\alpha$.
2. Under $H_1$ ($\rho_{AB} \neq \rho_{AC}$), the test detects the difference.
3. The straddle-zero p-value is biased at small N — visible in the rejection-rate gap.
4. The bootstrap diff distribution is correctly **paired** — share-of-resample preserves dependence.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

HERE = Path.cwd(); ROOT = HERE.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from python import (
    calculate_split_half_reliability,
    paired_bootstrap_compare_correlations,
    power_simulation_paired_bootstrap,
)

In [ ]:
def synth_three_groups(rho=(0.5, 0.5, 0.5), n=(25, 25, 25), n_items=50,
                       base_rate=0.7, item_spread=0.15, target_rel=0.7, seed=0):
    """Three groups with multivariate-normal latent item rates."""
    rng = np.random.default_rng(seed)
    R = np.array([[1, rho[0], rho[1]], [rho[0], 1, rho[2]], [rho[1], rho[2], 1]])
    Z = rng.multivariate_normal([0, 0, 0], R, size=n_items)
    item_rates = np.clip(base_rate + item_spread * Z, 0.02, 0.98)

    outs = []
    items = np.array([f'item{i:03d}' for i in range(n_items)], dtype=object)
    for g in range(3):
        if target_rel >= 1 or target_rel <= 0:
            sig_e = 0.0
        else:
            sig_e = item_spread * np.sqrt(n[g] * (1 - target_rel) / target_rel)
        E = sig_e * rng.standard_normal((n[g], n_items))
        P = np.clip(item_rates[:, g][None, :] + E, 0.01, 0.99)
        X = (rng.random((n[g], n_items)) < P).astype(float)
        outs.append(calculate_split_half_reliability(
            X, np.full_like(X, np.nan), items, n_splits=300, split_dim=1,
            rng=rng,
        ))
    return outs

## Single H0 run — all three true correlations equal

All three observed correlations should land close to each other, and all three p-values should be far from significant. The bootstrap diff distributions should be centered near 0.

In [ ]:
outs = synth_three_groups(rho=(0.5, 0.5, 0.5), n=(25, 25, 25), n_items=50, seed=7)
res = paired_bootstrap_compare_correlations(
    outs[0], outs[1], outs[2], trial_type='hit',
    n_boot=2000, min_resp=2, rng=np.random.default_rng(8),
)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3), sharey=True)
for ax, key, title in zip(axes, ['AB_minus_AC','AB_minus_BC','AC_minus_BC'],
                          ['r(A,B) - r(A,C)', 'r(A,B) - r(B,C)', 'r(A,C) - r(B,C)']):
    d = res.diffs[key]
    ax.hist(d, bins=40, color='#4060b0', alpha=0.7)
    ax.axvline(0, color='k', lw=1)
    ax.axvline(res.observed_diffs[key], color='r', lw=1.5, ls='--', label='observed')
    ax.set_title(title); ax.set_xlabel('diff')
    ax.legend()
axes[0].set_ylabel('count')
plt.suptitle('Paired-bootstrap diff distributions (H0: all rho equal)', y=1.02)
plt.tight_layout(); plt.show()

## H0 calibration sweep (multiple reps)

Under H0 the **recentered-null p-value** should reject at ~$\alpha=0.05$. The **straddle-zero p-value** is typically conservative at small N. Watch the gap.

In [ ]:
h0 = power_simulation_paired_bootstrap(
    n_reps=200, n_a=25, n_b=25, n_c=25, n_items=50,
    rho=(0.5, 0.5, 0.5),  # H0
    n_boot=500, alpha=0.05, seed=11, progress_every=50,
)

## H1 power — one pair differs by 0.25

We set $\rho_{AB}=0.65$ and $\rho_{AC}=\rho_{BC}=0.40$. The **AB vs AC** and **AB vs BC** comparisons should now reject more often; **AC vs BC** still reflects H0 (no true difference) and should reject at $\sim\alpha$.

The covariance matrix must remain PSD: with $\rho_{AC}=\rho_{BC}=0.40$ and $\rho_{AB}=0.65$, this is fine.

In [ ]:
h1 = power_simulation_paired_bootstrap(
    n_reps=200, n_a=25, n_b=25, n_c=25, n_items=50,
    rho=(0.65, 0.40, 0.40),  # AB > AC = BC
    n_boot=500, alpha=0.05, seed=12, progress_every=50,
)

In [ ]:
import pandas as pd
summary = pd.DataFrame({
    'pair': ['AB vs AC','AB vs BC','AC vs BC'],
    'H0 recentered-null': h0.reject_null.round(3),
    'H0 straddle-zero'  : h0.reject_straddle.round(3),
    'H1 recentered-null': h1.reject_null.round(3),
    'H1 straddle-zero'  : h1.reject_straddle.round(3),
})
print(summary.to_string(index=False))
print()
print('Expected: H0 ~ 0.05 for both p-values (recentered-null more accurate).')
print('Expected: H1 high for AB-vs-AC and AB-vs-BC, low (~0.05) for AC-vs-BC.')

**Interpretation.**

- If the recentered-null rate under H0 is ~0.05, the test is well-calibrated at this N.
- If the straddle-zero rate under H0 is much lower than 0.05, that confirms the small-N bias — straddle-zero is conservative under skewed bootstrap diff distributions.
- Under H1, the AC-vs-BC row should look like H0 (no true difference); the other two rows should reject most of the time.

**Next:** notebook 04 shows how to calibrate at *your* actual site Ns before reporting any significant difference.